# PT-W3-D5 概念实验：Ontology 驱动 Digital Employee

数字员工是 Context、Capability、Policy 和制品引用的语义锚点，不是把 Prompt、知识、运行时状态塞进一个巨型对象。

## 实验 1：编译并验证数字员工定义

验证 Cognitive Scope、Capability Set、Policy Bundle 与 Knowledge Scope 是否都落在企业 Ontology 声明的边界内。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import font_manager
font_path = '/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc'
font_manager.fontManager.addfont(font_path)
font_name = font_manager.FontProperties(fname=font_path).get_name()
plt.rcParams['font.family'] = font_name
plt.rcParams['axes.unicode_minus'] = False

ONTOLOGY = {
    'contexts': {'Leasing', 'Contract', 'Merchant', 'Billing', 'CustomerService'},
    'capabilities': {'query_space': 'Leasing', 'lock_space': 'Leasing', 'read_contract': 'Contract', 'create_ticket': 'CustomerService'},
    'knowledge': {'leasing_policy': 'Leasing', 'brand_catalog': 'Merchant', 'contract_terms': 'Contract', 'finance_report': 'Billing'},
}
EMPLOYEE = {
    'id': 'de-leasing-ops', 'version': 1, 'lifecycle': 'Published',
    'cognitive_scope': {'Leasing', 'Contract', 'Merchant'},
    'capabilities': {'query_space', 'lock_space', 'read_contract'},
    'policies': {'query_space': 'read_only', 'lock_space': 'conditional_write', 'read_contract': 'read_only'},
    'knowledge_scope': {'leasing_policy', 'brand_catalog', 'contract_terms'},
    'references': {'ApplicationContractVersion': 'acv:7b2', 'BlueprintVersion': 'bpv:19a'},
}
print('数字员工语义锚点:', EMPLOYEE['id'], EMPLOYEE['references'])

In [ ]:
def validate_employee(employee):
    errors = []
    for cap in employee['capabilities']:
        owner = ONTOLOGY['capabilities'].get(cap)
        if owner is None: errors.append(f'能力不存在: {cap}')
        elif owner not in employee['cognitive_scope']: errors.append(f'能力越界: {cap} 属于 {owner}')
        elif cap not in employee['policies']: errors.append(f'能力缺 Policy: {cap}')
    for doc in employee['knowledge_scope']:
        owner = ONTOLOGY['knowledge'].get(doc)
        if owner not in employee['cognitive_scope']: errors.append(f'知识越界: {doc} 属于 {owner}')
    return {'valid': not errors, 'errors': errors, 'artifact_refs': employee['references']}

result = validate_employee(EMPLOYEE)
print('合法定义:', result)
broken = dict(EMPLOYEE, knowledge_scope=EMPLOYEE['knowledge_scope'] | {'finance_report'})
print('注入财务知识后的校验:', validate_employee(broken))

In [ ]:
groups = ['认知 Context', '能力', 'Policy', '知识', '制品引用']
counts = [len(EMPLOYEE['cognitive_scope']), len(EMPLOYEE['capabilities']), len(EMPLOYEE['policies']), len(EMPLOYEE['knowledge_scope']), len(EMPLOYEE['references'])]
fig, ax = plt.subplots(figsize=(6, 3))
ax.bar(groups, counts, color=['#457b9d', '#2a9d8f', '#e9c46a', '#f4a261', '#8d99ae'])
ax.set_title('数字员工的声明式语义构成'); ax.set_ylabel('声明数量')
ax.tick_params(axis='x', rotation=20)
plt.tight_layout(); plt.show()